# Backfill 2025 data missed by the original migration

The first migration (`script.ipynb`) intentionally narrowed two things to keep the initial import small:

- **Guest sessions** (`presences` with `membre` null / `prix` set) were only imported from **2026-01-01** onward.
- **Members** with no currently-active subscription were only picked up if their *latest* inscription started on/after **2025-08-01**.

That left a gap in `payments`/revenue history for 2025. This notebook re-reads the same legacy `gym` database and fills that gap:

- Guest sessions from **2025-01-01 to 2025-12-31**.
- Members (and their historical payments) whose latest inscription falls between **2025-01-01 and 2025-07-31**.

Every insert below reuses the same dedup rules as the original migration (skip a member already present by `rfid_uid`, skip a payment that already matches) plus a new guard on guest sessions, so this notebook is safe to run even though the live database already has the 2026 guest sessions and Aug-2025-onward members in it -- and safe to re-run if interrupted.

In [12]:
import mariadb

conn = mariadb.connect(
    host="localhost",
    port=3306,
    user="root",
    password="",
    database="gym"
)

cursor = conn.cursor()

In [13]:
import pandas as pd
pd.set_option('display.max_columns', None)

df_membres = pd.read_sql("SELECT * FROM membres", conn)
df_inscriptions = pd.read_sql("SELECT * FROM inscriptions", conn)
df_presences = pd.read_sql("SELECT * FROM presences", conn)

conn.close()

C:\Users\zakmins\AppData\Local\Temp\ipykernel_11052\1939934059.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_membres = pd.read_sql("SELECT * FROM membres", conn)
C:\Users\zakmins\AppData\Local\Temp\ipykernel_11052\1939934059.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_inscriptions = pd.read_sql("SELECT * FROM inscriptions", conn)
C:\Users\zakmins\AppData\Local\Temp\ipykernel_11052\1939934059.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_presences = pd.read_sql("SELECT * FROM presences", conn)


## Guest sessions: fill in 2025-01-01 -> 2025-12-31

Same filter as the original notebook (`membre` is null, `prix` is set = a walk-in paid visit not tied to a member), but instead of a lower-bound-only cutoff at 2026-01-01, this pulls the full 2025 range that was skipped the first time around.

In [14]:
df_guest_presences = df_presences.loc[
    df_presences['membre'].isna() &
    df_presences['prix'].notna()
].copy()

GUEST_SESSIONS_START = '2025-01-01'
GUEST_SESSIONS_END = '2026-01-01'  # exclusive - everything from here on was already imported

df_guest_presences = df_guest_presences[
    (df_guest_presences['created_at'] >= GUEST_SESSIONS_START) &
    (df_guest_presences['created_at'] < GUEST_SESSIONS_END)
]
df_guest_presences['prix'] = df_guest_presences['prix'].round().astype(int)

print(f"{len(df_guest_presences)} guest session(s) between {GUEST_SESSIONS_START} and {GUEST_SESSIONS_END}")
df_guest_presences.head()

4169 guest session(s) between 2025-01-01 and 2026-01-01


,id,inscription,membre,created_at,updated_at,type,prix,activity,telephone,nom_prenom,user
24,28,NaN,None,2025-01-04 15:43:22,2025-01-04 15:43:22,1.0,300,None,None,MAHREZ,53.0
27,31,NaN,None,2025-01-04 17:26:36,2025-01-04 17:26:36,1.0,300,None,None,JARI,53.0
28,32,NaN,None,2025-01-04 18:17:41,2025-01-04 18:17:41,1.0,300,None,None,BOUALI,53.0
31,35,NaN,None,2025-01-04 18:48:13,2025-01-04 18:48:13,1.0,300,None,None,BOUALI ADLEN,53.0
32,36,NaN,None,2025-01-04 18:54:45,2025-01-04 18:54:45,1.0,300,None,None,LEKHAL,53.0


## Write the missing guest sessions into the live Smolympic SQLite database

Same mapping as the original migration:

| df_guest_presences column | Smolympic field |
|---|---|
| `nom_prenom` | `guest_sessions.name` (falls back to `"Invite"` if blank/null) |
| `prix` (rounded to int) | `guest_sessions.amount` **and** `payments.amount` |
| `created_at` | `guest_sessions.entry_time` **and** `payments.date` |
| -- | `guest_sessions.expires_at` = `entry_time` + 2h |

Same `payments` mirroring as the live "+ Session" walk-in flow (`kind='session'`, `sport='GYM'`, `method='Cash'`, `member_id=NULL`, `walk_in=1`).

Unlike the original cell, this one **does** guard against re-running: it skips any row whose `(name, amount, entry_time)` already exists in `guest_sessions`, so running this notebook twice (or after a partial failure) won't double-insert.

**Close the Smolympic app first.**

In [15]:
import os, shutil, sqlite3
from datetime import datetime, timedelta

db_path = os.path.expandvars(r'%APPDATA%\SMOLYMPIC\smolympic.db')
assert os.path.exists(db_path), f"Smolympic database not found at {db_path}"

backup_path = f"{db_path}.bak-{datetime.now():%Y%m%d%H%M%S}"
shutil.copy2(db_path, backup_path)
print(f"Backed up live database to {backup_path}")


def clean_str(v):
    if v is None or pd.isna(v):
        return None
    v = str(v).strip()
    return v or None


def iso(dt):
    return dt.strftime('%Y-%m-%dT%H:%M:%S')


conn = sqlite3.connect(db_path)
conn.execute('PRAGMA foreign_keys = ON')
cur = conn.cursor()

existing_sessions = {
    (r[0], r[1], r[2])
    for r in cur.execute('SELECT name, amount, entry_time FROM guest_sessions')
}

inserted, skipped_dupe, fallback_named = 0, 0, 0

try:
    for _, row in df_guest_presences.iterrows():
        name = clean_str(row['nom_prenom'])
        if not name:
            name = 'Invite'
            fallback_named += 1

        amount = int(row['prix'])
        entry_iso = iso(row['created_at'])

        key = (name, amount, entry_iso)
        if key in existing_sessions:
            skipped_dupe += 1
            continue

        expires_iso = iso(row['created_at'] + timedelta(hours=2))

        cur.execute(
            'INSERT INTO guest_sessions (name,amount,entry_time,expires_at) VALUES (?,?,?,?)',
            (name, amount, entry_iso, expires_iso),
        )
        cur.execute(
            'INSERT INTO payments (member_id,amount,kind,sport,method,date,walk_in) VALUES (NULL,?,?,?,?,?,1)',
            (amount, 'session', 'GYM', 'Cash', entry_iso),
        )
        existing_sessions.add(key)
        inserted += 1
except Exception:
    conn.rollback()
    conn.close()
    raise

conn.commit()
conn.close()

print(f"Inserted {inserted} guest session(s) (+ matching payment rows).")
if skipped_dupe:
    print(f"Skipped {skipped_dupe} row(s) already present in guest_sessions (likely a prior run of this cell).")
if fallback_named:
    print(f"{fallback_named} row(s) had no nom_prenom and were named 'Invite'.")
print(f"A backup of the pre-import database was saved to: {backup_path}")

Backed up live database to C:\Users\zakmins\AppData\Roaming\SMOLYMPIC\smolympic.db.bak-20260823215540
Inserted 4167 guest session(s) (+ matching payment rows).
Skipped 2 row(s) already present in guest_sessions (likely a prior run of this cell).
2758 row(s) had no nom_prenom and were named 'Invite'.
A backup of the pre-import database was saved to: C:\Users\zakmins\AppData\Roaming\SMOLYMPIC\smolympic.db.bak-20260823215540


## Members: extend the "still recent" cutoff back to 2025-01-01

The original migration built `df_combined` from two groups:

1. Every member with a currently-active (`etat=1`) inscription, any date.
2. Members with no active inscription, but whose *latest* inscription started on/after 2025-08-01 and is `etat=0` -- added so recently-lapsed members weren't lost.

Group 2's cutoff means a member whose subscription lapsed for good sometime in **2025-01-01 to 2025-07-31** -- and who never resubscribed -- was never imported, so their historical payments are missing from revenue reports. This section rebuilds the same two groups with the cutoff moved back to 2025-01-01.

Group 1 is recomputed identically (it isn't date-bounded), so it's unchanged. Group 2 now includes both the members it caught the first time *and* the new 2025-01-01 -> 2025-07-31 ones -- that's fine: the member-insert cell below already skips anyone whose `matricule` is already an `rfid_uid` in the live database, so already-migrated members are silently skipped and only the new ones get inserted.

In [16]:
df_membres.drop(columns=['photo', 'etat', 'email', 'identite', 'type', 'source', 'cn', 'dm', 'remarque', 'created_at', 'updated_at'], inplace=True)

df_inscriptions.drop(columns=['type', 'remarque', 'activities', 'assurance', 'user', 'tripode', 'see', 'created_at', 'updated_at'], inplace=True)
df_inscriptions['membre'] = df_inscriptions['membre'].astype(int)

In [17]:
df_combined = pd.merge(
    df_membres,
    df_inscriptions[df_inscriptions['etat'] == '1'],
    left_on='id',
    right_on='membre',
    how='inner'
)

In [18]:
df_inscriptions['debut'] = pd.to_datetime(df_inscriptions['debut'], format='%Y-%m-%d %H:%M:%S')

In [19]:
MEMBERS_THRESHOLD_DATE = '2025-01-01'

recent = df_inscriptions[df_inscriptions['debut'] >= MEMBERS_THRESHOLD_DATE]
latest_recent = recent.sort_values('debut').groupby('membre', as_index=False).tail(1)
inactive_latest = latest_recent[latest_recent['etat'] == '0']

df_inactive = pd.merge(
    df_membres,
    inactive_latest,
    left_on='id',
    right_on='membre',
    how='inner',
)
candidates = len(df_inactive)
df_inactive = df_inactive[~df_inactive['id_x'].isin(df_combined['id_x'])]
print(f"Adding {len(df_inactive)} inactive member(s) whose latest subscription since {MEMBERS_THRESHOLD_DATE} is etat=0 "
      f"({candidates - len(df_inactive)} already covered by df_combined and left untouched).")

df_combined = pd.concat([df_combined, df_inactive], ignore_index=True)

Adding 1167 inactive member(s) whose latest subscription since 2025-01-01 is etat=0 (1 already covered by df_combined and left untouched).


In [20]:
df_combined['assurance'] = df_combined['assurance'].fillna(0).astype(int)

## Write df_combined into the live Smolympic SQLite database

Identical mapping and rules as the original migration's member-import cell:

| df_combined column(s) | Smolympic field |
|---|---|
| `nom`, `prenom` | `name` (title-cased, "Prenom Nom") |
| `sexe` (`homme`/`femme`) | `gender` (`M`/`F`) |
| `naissance` | `dob` |
| `telephone` | `phone` |
| `matricule` | `rfid_uid`, left-padded with zeros to 10 digits |
| `sang` | `blood_type` |
| `assurance` | `insurance` (+ `insurance_expiry` = sub_start + 365d) |
| `debut`, `fin` | `sub_start`, `sub_end` |
| `nbrmois` | `duration_days` (x 30) |
| `nbsseance`, `reste` | `sessions_total`, `sessions_left` |
| `versement` | recorded as a `payments` row |
| -- | `balance` = 0 for every imported member |

**Safe to re-run**: a row whose `matricule` already exists as an `rfid_uid` is skipped -- this is what makes it safe that `df_combined` here also contains the members from the original Aug-2025-onward run.

**Close the Smolympic app first.**

In [21]:
import os, shutil, sqlite3, random
from datetime import datetime, timedelta

INSURANCE_DAYS = 365
GENDER_MAP = {'homme': 'M', 'femme': 'F'}
BLOOD_TYPES = {'A+', 'A-', 'B+', 'B-', 'AB+', 'AB-', 'O+', 'O-'}

db_path = os.path.expandvars(r'%APPDATA%\SMOLYMPIC\smolympic.db')
assert os.path.exists(db_path), f"Smolympic database not found at {db_path}"

backup_path = f"{db_path}.bak-{datetime.now():%Y%m%d%H%M%S}"
shutil.copy2(db_path, backup_path)
print(f"Backed up live database to {backup_path}")


def clean_str(v):
    if v is None or pd.isna(v):
        return None
    v = str(v).strip()
    return v or None


def date_only(v):
    if v is None or pd.isna(v):
        return None
    return v.strftime('%Y-%m-%d') if hasattr(v, 'strftime') else str(v)[:10]


def timestamp(date_str):
    return f"{date_str}T00:00:00" if date_str else datetime.now().strftime('%Y-%m-%dT%H:%M:%S')


def clean_num(v, default=0):
    if v is None or pd.isna(v):
        return default
    return float(v)


conn = sqlite3.connect(db_path)
conn.execute('PRAGMA foreign_keys = ON')
cur = conn.cursor()

existing_count = cur.execute('SELECT COUNT(*) FROM members').fetchone()[0]
if existing_count:
    print(f"members table already has {existing_count} row(s) - importing on top of existing data.")

already_in_db = {r[0] for r in cur.execute('SELECT rfid_uid FROM members')}
existing_rfids = set(already_in_db)
_rfid_seq = 0


def next_placeholder_rfid():
    global _rfid_seq
    while True:
        _rfid_seq += 1
        candidate = str(4200000000 + _rfid_seq * 13)
        if candidate not in existing_rfids:
            return candidate


dup_ids = df_combined.loc[df_combined['id_x'].duplicated(), 'id_x'].tolist()
if dup_ids:
    print(f"Warning: {len(dup_ids)} member id(s) appear more than once in df_combined: {dup_ids}")

inserted, skipped, bad_gender, already_imported = 0, [], 0, []
missing_rfid, dup_rfid = [], []

try:
    for _, row in df_combined.iterrows():
        matricule_rfid = clean_str(row['matricule'])
        if matricule_rfid:
            matricule_rfid = matricule_rfid.zfill(10)
        if matricule_rfid and matricule_rfid in already_in_db:
            already_imported.append(row['id_x'])
            continue

        nom, prenom = clean_str(row['nom']), clean_str(row['prenom'])
        if not nom and not prenom:
            skipped.append(row['id_x'])
            continue
        name = ' '.join(p.title() for p in (prenom, nom) if p)

        sexe_key = (clean_str(row['sexe']) or '').lower()
        if sexe_key not in GENDER_MAP:
            bad_gender += 1
        gender = GENDER_MAP.get(sexe_key, 'M')

        dob = date_only(row['naissance'])
        phone = clean_str(row['telephone'])

        blood = clean_str(row['sang'])
        if blood:
            blood = blood.upper()
            if blood not in BLOOD_TYPES:
                blood = None

        if matricule_rfid and matricule_rfid not in existing_rfids:
            rfid = matricule_rfid
        else:
            (dup_rfid if matricule_rfid else missing_rfid).append(row['id_x'])
            rfid = next_placeholder_rfid()
        existing_rfids.add(rfid)

        sub_start = date_only(row['debut'])
        sub_end = date_only(row['fin'])
        months = int(clean_num(row['nbrmois'], 1)) or 1
        duration_days = months * 30

        sessions_total = None if pd.isna(row['nbsseance']) else int(row['nbsseance'])
        sessions_left = None if pd.isna(row['reste']) else int(row['reste'])

        paid = max(0, round(clean_num(row['versement'])))
        balance = 0

        insured = bool(clean_num(row['assurance']))
        insurance_expiry = None
        if insured:
            anchor = sub_start or datetime.now().strftime('%Y-%m-%d')
            insurance_expiry = timestamp(
                (datetime.strptime(anchor, '%Y-%m-%d') + timedelta(days=INSURANCE_DAYS)).strftime('%Y-%m-%d')
            )

        join_date = timestamp(sub_start)
        hue = random.randint(0, 359)

        cur.execute(
            "INSERT INTO members (rfid_uid,name,gender,dob,phone,blood_type,sports,membership_type,"
            "sub_start,sub_end,duration_days,sessions_total,sessions_left,"
            "insurance,insurance_expiry,balance,hue,join_date) "
            "VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
            (rfid, name, gender, dob, phone, blood, '["GYM"]', 'subscription',
             sub_start, sub_end, duration_days, sessions_total, sessions_left,
             int(insured), insurance_expiry, balance, hue, join_date),
        )
        member_id = cur.lastrowid

        if paid > 0:
            cur.execute(
                "INSERT INTO payments (member_id,amount,kind,sport,method,date) VALUES (?,?,?,?,?,?)",
                (member_id, paid, 'subscription', 'GYM', 'Cash', timestamp(sub_start)),
            )
        inserted += 1
except Exception:
    conn.rollback()
    conn.close()
    raise

conn.commit()
conn.close()

print(f"Inserted {inserted} member(s).")
if already_imported:
    print(f"Skipped {len(already_imported)} row(s) already migrated in a previous run (old id_x): {already_imported}")
if skipped:
    print(f"Skipped {len(skipped)} row(s) with no name (old id_x): {skipped}")
if bad_gender:
    print(f"{bad_gender} row(s) had an unrecognized 'sexe' value and were defaulted to gender='M'.")
if missing_rfid:
    print(f"{len(missing_rfid)} row(s) had no matricule and got a generated placeholder tag (old id_x): {missing_rfid}")
if dup_rfid:
    print(f"{len(dup_rfid)} row(s) had a matricule already used elsewhere and got a generated placeholder tag instead (old id_x): {dup_rfid}")
print("Note: 'adresse' has no matching field in Smolympic and was not imported.")
print(f"A backup of the pre-import database was saved to: {backup_path}")

Backed up live database to C:\Users\zakmins\AppData\Roaming\SMOLYMPIC\smolympic.db.bak-20260823215541
members table already has 1082 row(s) - importing on top of existing data.
Inserted 268 member(s).
Skipped 1080 row(s) already migrated in a previous run (old id_x): [8, 12, 14, 16, 36, 47, 58, 77, 786, 100, 103, 128, 134, 135, 978, 142, 155, 169, 171, 177, 178, 179, 181, 199, 206, 234, 235, 255, 264, 278, 304, 310, 322, 336, 371, 393, 395, 403, 414, 428, 429, 434, 461, 477, 472, 506, 533, 534, 538, 539, 546, 550, 551, 569, 572, 586, 587, 605, 611, 613, 623, 627, 683, 687, 693, 711, 726, 728, 738, 748, 774, 819, 830, 835, 863, 899, 921, 954, 960, 1371, 976, 984, 1006, 1012, 1024, 1025, 1041, 1044, 1046, 1048, 1050, 1063, 1084, 1111, 1112, 1137, 1154, 1161, 1162, 1163, 1339, 1173, 1174, 1179, 1194, 1203, 1211, 1215, 1231, 1233, 1240, 1243, 1244, 1245, 1246, 1253, 1260, 1267, 1269, 1277, 1283, 1284, 1285, 1288, 1290, 1291, 1293, 1316, 1317, 1318, 1320, 1321, 1322, 1323, 1324, 1326, 1327,

## Backfill revenue history: one payment per inscription, not per member

Same rationale as the original migration's backfill cell: `df_combined` keeps at most one inscription per member, so every *other* inscription that member ever had (each with its own `versement`) needs its own `payments` row for revenue reports to be accurate.

For every member in `df_combined` above (now including the 2025-01-01 -> 2025-07-31 additions) that has a real `rfid_uid` in the live database, this walks *every* row in `df_inscriptions` belonging to them and inserts a `payments` row for each one not already covered -- skipping any inscription for which a matching `payments` row already exists (`member_id`, `amount`, `date`), so it's safe to re-run and safe that it also re-scans the members from the original Aug-2025-onward run.

Backs up the database first.

In [ ]:
import os, shutil, sqlite3
from datetime import datetime

db_path = os.path.expandvars(r'%APPDATA%\SMOLYMPIC\smolympic.db')
assert os.path.exists(db_path), f"Smolympic database not found at {db_path}"

backup_path = f"{db_path}.bak-{datetime.now():%Y%m%d%H%M%S}"
shutil.copy2(db_path, backup_path)
print(f"Backed up live database to {backup_path}")


def clean_str(v):
    if v is None or pd.isna(v):
        return None
    v = str(v).strip()
    return v or None


def clean_num(v, default=0):
    if v is None or pd.isna(v):
        return default
    return float(v)


def date_only(v):
    if v is None or pd.isna(v):
        return None
    return v.strftime('%Y-%m-%d') if hasattr(v, 'strftime') else str(v)[:10]


def timestamp(date_str):
    return f"{date_str}T00:00:00" if date_str else None


conn = sqlite3.connect(db_path)
cur = conn.cursor()

rfid_to_member_id = {r[0]: r[1] for r in cur.execute('SELECT rfid_uid, id FROM members')}

existing_payments = {
    (r[0], r[1], r[2])
    for r in cur.execute("SELECT member_id, amount, date FROM payments WHERE kind='subscription'")
}

inserted, already_present, no_versement, unmatched_member = 0, 0, 0, []

for _, row in df_combined.iterrows():
    matricule = clean_str(row['matricule'])
    if not matricule:
        continue
    rfid = matricule.zfill(10)
    member_id = rfid_to_member_id.get(rfid)
    if member_id is None:
        unmatched_member.append(row['id_x'])
        continue

    already_covered_inscription_id = row['id_y']
    member_inscriptions = df_inscriptions[df_inscriptions['membre'] == row['id_x']]

    for _, insc in member_inscriptions.iterrows():
        if insc['id'] == already_covered_inscription_id:
            continue

        amount = max(0, round(clean_num(insc['versement'])))
        if amount <= 0:
            no_versement += 1
            continue

        date_str = timestamp(date_only(insc['debut']))
        key = (member_id, amount, date_str)
        if key in existing_payments:
            already_present += 1
            continue

        cur.execute(
            "INSERT INTO payments (member_id,amount,kind,sport,method,date) VALUES (?,?,?,?,?,?)",
            (member_id, amount, 'subscription', 'GYM', 'Cash', date_str),
        )
        existing_payments.add(key)
        inserted += 1

conn.commit()
conn.close()

print(f"Inserted {inserted} backfilled payment(s) from prior inscriptions.")
if already_present:
    print(f"{already_present} inscription(s) skipped - a matching payment already existed (likely a prior run of this cell or the original migration).")
if no_versement:
    print(f"{no_versement} inscription(s) had no versement (0 or missing) and were skipped.")
if unmatched_member:
    print(f"{len(unmatched_member)} member(s) could not be matched to a real rfid_uid (placeholder-tagged) and were skipped (old id_x): {unmatched_member}")

Backed up live database to C:\Users\zakmins\AppData\Roaming\SMOLYMPIC\smolympic.db.bak-20260823215541
Inserted 110 backfilled payment(s) from prior inscriptions.
1903 inscription(s) skipped - a matching payment already existed (likely a prior run of this cell or the original migration).
14 inscription(s) had no versement (0 or missing) and were skipped.


: 